# FinBERT Sentiment Analysis
Neural approach using . Runs evaluation and logs results to MLflow.

In [ ]:
import sys
sys.path.insert(0, "..")

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from src.finbert.model import load_pipeline, predict
from src.utils.metrics import compute_metrics


In [ ]:
# Run evaluation — logs params, metrics, artifacts and registers model in MLflow
import subprocess
result = subprocess.run(
    [sys.executable, "../src/finbert/evaluate.py", "--dataset", "../data/data.csv"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


In [ ]:
# Load latest FinBERT run from MLflow
mlflow.set_tracking_uri("../mlruns")
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("financial-sentiment")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="params.approach = 'finbert'",
    order_by=["start_time DESC"],
    max_results=1,
)
run = runs[0]
print("Params:", run.data.params)
print("Metrics:", {k: round(v, 4) for k, v in run.data.metrics.items()})


In [ ]:
# Compare all approaches side by side
all_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)
comparison = pd.DataFrame([
    {
        "approach": r.data.params.get("approach"),
        "dataset": r.data.params.get("dataset"),
        "accuracy": round(r.data.metrics.get("accuracy", 0), 4),
        "f1_macro": round(r.data.metrics.get("f1_macro", 0), 4),
    }
    for r in all_runs
])
comparison
